**Instruções iniciais**

*   Abra os links dos dados:
    * https://tinyurl.com/bigdata-mcu
*   Clique em "Adicionar atalho ao Drive"


# Solução

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Configuração do ambiente

In [2]:
!pip install pyspark

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import Row

from datetime import datetime

appName = 'Big Data'
master = 'local[*]'

spark = SparkSession.builder     \
    .master(master) \
    .appName(appName) \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

## Leitura de dados

In [4]:
# Usar esta entrada para testes
input_data = spark.sparkContext.textFile('file:///content/drive/My Drive/mcu/mcu_subset.csv')

In [5]:
# Usar esta entrada para entrega final
# input_data = spark.sparkContext.textFile('file:///content/drive/My Drive/mcu/mcu.csv')

## Exemplo de uso do pipeline

In [6]:
from transformers import pipeline

# Baixar e configurar pipeline do modelo
sentiment = pipeline('sentiment-analysis')



[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

In [7]:
result = sentiment("I am Iron Man")

In [8]:
result

[{'label': 'POSITIVE', 'score': 0.999076247215271}]

In [9]:
result[0]['label']

'POSITIVE'

## Solução

In [10]:
# Inclua outros personagens de sua escolha
characters = {'tony stark', 'steve rogers', 'thanos', 'bruce banner'}

In [11]:
import re

# Modifique a solução para implementar a função Map
def line_sentiment(line) :
  #line = re.sub('[^a-záàâãéêíóôõúç ]', ' ', line.lower())
  fields = line.split(';')

  personagem = fields[1].lower()
  if personagem in characters:
    line = fields[2]
    result = sentiment(line)
    polaridade = 1 if result[0]['label'] == 'POSITIVE' else -1
    contagem = 1
    yield (personagem, (polaridade, contagem))

In [19]:
s = input_data.flatMap(line_sentiment)

In [20]:
s.take(10)

[('tony stark', (-1, 1)),
 ('tony stark', (-1, 1)),
 ('tony stark', (1, 1)),
 ('tony stark', (-1, 1)),
 ('steve rogers', (-1, 1)),
 ('steve rogers', (-1, 1)),
 ('bruce banner', (1, 1)),
 ('bruce banner', (1, 1)),
 ('steve rogers', (-1, 1)),
 ('tony stark', (-1, 1))]

In [14]:
s.count()

33

In [21]:
def sentimentTotal(acc, v):
  return (v[0]+acc[0], acc[1] + v[1])

s = s.reduceByKey(sentimentTotal)

In [23]:
# Implemente e aplique um método para calculo do sentimento médio
def sentimentAvg(x):
  return x[0] / x[1]

s.mapValues(sentimentAvg).collect()

[('tony stark', -0.4666666666666667),
 ('thanos', 1.0),
 ('bruce banner', 0.14285714285714285),
 ('steve rogers', -1.0)]

# Resultado Final


Apresente o resultado final da sua análise completa.